In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

# result

> Result class and all methods to get details and process the publications

In [ ]:
#| default_exp result

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import logging
from fastcore.all import *
from typing import Dict, List
from pydantic import BaseModel
from datetime import date, datetime
import pandas as pd


In [ ]:
#| hide
from dotenv import load_dotenv, find_dotenv


In [ ]:
#| hide
logger = logging.getLogger(__name__)

In [ ]:
#| hide
load_dotenv('pass.env')

False

# Fetching some results from search. Should change to a class soon

In [ ]:
#| export
from pubmed_lib.parser import *
from pubmed_lib.author import *

In [ ]:
#| exports
class Result(BaseModel):
    """
    Model for results, including all the data taht will be retreived and their type
    """
    pubmed:str
    pmc: str | None = None
    doi: str | None = None
    pii: str | None = None
    abstract: str
    autorlist: List[Autor]
    title: str
    journal: str
    published: date | None = None 
    mayorKeys: List[str]
    mayorMesh: List[str]
    minorMesh: List[str]
          
        

In [ ]:
#| export
@patch
def __eq__(
    self:Result,
    other:Result
) -> bool:
    if isinstance(other, Result):
        return self.pubmed == other.pubmed
    return False

In [ ]:
#| export
@patch
def __neq__(
    self:Result,
    other:Result
) -> bool:
    if isinstance(other, Result):
        return self.pubmed != other.pubmed
    return False

# Class for list of `Result` and its methods

In [ ]:
#| exports

class Results(BaseModel):
    """Class to store a list of `Result`"""
    results: List[Result] =[]

In [ ]:
#| export

@patch
def append(
    self:Results,
    item: Result
)->None:
    """Method to add item on the list of result"""
    self.results.append(item)

In [ ]:
#| export

@patch
def _df(
    self:Results,
)->None:
    """Method to add item on the list of result"""
    return pd.DataFrame([x.model_dump() for x in self.results])

In [ ]:
#| export

@patch
def to_df(
    self:Results,
    mode: str = 'publication'
)->None:
    """Method to add item on the list of result"""
    df = self._df()
    match mode:
        case 'publication': 
            return df
        case 'exploded':
            return df.explode('autorlist')
        case 'autors':
            df_e = df.explode('autorlist').reset_index()
            autors = pd.DataFrame(df_e['autorlist'].to_list())
            return pd.merge( 
                            autors,
                            df_e,
                            left_index=True, 
                            right_index=True 
                            ).drop(columns=['affiliation_parsed','autorlist'])[['name','affiliations', 'identifier', 'email', 
       'organization', 'laboratory', 'department', 'faculty', 'country',
       'city', 'state', 'pubmed',  'doi', 
       'abstract', 'title', 'journal', 'published']]


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()